# 03 - MPC Geometry: Feasibility, Terminal Sets, Stability

This notebook is about the geometry behind the terminal ingredients in Model Predictive Control (MPC).

The previous notebooks introduced LQR, saturated feedback, and finite-horizon MPC formulations. Here we ask a different question:

> What do terminal sets, recursive feasibility, and stability look like?

The examples are intentionally small. They are visual teaching examples, not a replacement for proofs. Whenever we simulate a controller, the simulation illustrates the mechanism. The proof requires invariance and Lyapunov arguments.

## 1. Why another notebook?

MPC is not only an optimization problem.

A finite-horizon optimizer can produce useful control inputs, but guarantees usually require more structure:

- a **feasible set**: states from which the finite-horizon problem has an admissible solution;
- a **terminal set**: a safe target region at the end of the prediction horizon;
- a **terminal controller**: a simple local feedback law that keeps the system safe inside the terminal set;
- a **terminal cost**: a local value function that gives the finite-horizon problem the right stabilizing geometry.

This notebook visualizes those ideas using only scalar and two-dimensional systems.

In [ ]:
%matplotlib inline

import numpy as np
import matplotlib.pyplot as plt
from scipy.linalg import solve_discrete_are
from scipy.optimize import minimize

plt.rcParams.update({
    "figure.figsize": (8, 4.5),
    "axes.grid": True,
    "grid.alpha": 0.25,
    "lines.linewidth": 2.0,
})

## 2. Scalar feasibility example

Consider the scalar constrained system

\begin{align}
x_{k+1} &= x_k + u_k,\\
|u_k| &\le 1,\\
|x_k| &\le 5,
\end{align}

with terminal set

\begin{align}
X_f = [-1, 1].
\end{align}

The $N$-step feasible set is the set of states that can be driven into $X_f$ in $N$ steps while respecting the constraints. For this scalar system, each predecessor set is still an interval.

In [ ]:
x_limit = 5.0
u_limit = 1.0
terminal = (-1.0, 1.0)

intervals = [terminal]
for _ in range(4):
    lower, upper = intervals[-1]
    predecessor = (max(-x_limit, lower - u_limit), min(x_limit, upper + u_limit))
    intervals.append(predecessor)

fig, ax = plt.subplots(figsize=(8, 4.6))

ax.axvspan(-x_limit, x_limit, color="0.92", label=r"state constraint $|x| \leq 5$")
ax.axvspan(terminal[0], terminal[1], color="tab:green", alpha=0.20, label="terminal set $X_f$")

for N, (lower, upper) in enumerate(intervals):
    ax.plot([lower, upper], [N, N], marker="|", markersize=18, linewidth=10, label=f"N = {N}")
    ax.text(upper + 0.12, N, f"[{lower:.0f}, {upper:.0f}]", va="center")

ax.set_xlim(-5.6, 5.6)
ax.set_ylim(-0.7, 4.7)
ax.set_xlabel("state $x$")
ax.set_ylabel("horizon $N$")
ax.set_title("N-step feasible intervals for the scalar integrator")
ax.set_yticks(range(5))
ax.legend(loc="lower right", fontsize=8)
plt.show()

Longer horizons make more states feasible, because the controller has more time to move the state toward the terminal set. The terminal set gives the finite-horizon problem a safe target instead of merely asking it to survive until the end of the prediction window.

## 3. Recursive feasibility via shifted input sequence

Recursive feasibility asks:

> If the MPC problem is feasible now, why should it still be feasible after applying the first input?

The usual proof mechanism constructs a candidate solution for the next time step. Start with a predicted feasible trajectory

\begin{align}
x_0, x_1, \ldots, x_N
\end{align}

and predicted inputs

\begin{align}
u_0, u_1, \ldots, u_{N-1}.
\end{align}

After applying $u_0$, the next optimizer can try the shifted candidate sequence

\begin{align}
u_1, \ldots, u_{N-1}, Kx_N.
\end{align}

This works when $X_f$ is invariant under the local terminal controller $u = Kx$.

In [ ]:
N = 4
K_terminal = -0.5

x_pred = np.array([3.4, 2.6, 1.8, 1.2, 0.8])
u_pred = np.diff(x_pred)

x_after_apply = x_pred[1]
u_shifted = np.r_[u_pred[1:], K_terminal * x_pred[-1]]
x_shifted = [x_after_apply]
for u in u_shifted:
    x_shifted.append(x_shifted[-1] + u)
x_shifted = np.array(x_shifted)

fig, (ax_x, ax_u) = plt.subplots(2, 1, figsize=(8, 6.2), sharex=False)

ax_x.axhspan(terminal[0], terminal[1], color="tab:green", alpha=0.18, label="$X_f$")
ax_x.axhline(x_limit, color="0.25", linestyle="--", label="state limits")
ax_x.axhline(-x_limit, color="0.25", linestyle="--")
ax_x.plot(np.arange(N + 1), x_pred, "o-", label="original prediction")
ax_x.plot(np.arange(1, N + 2), x_shifted, "s--", label="shifted candidate")
ax_x.set_xlabel("time index")
ax_x.set_ylabel("state $x$")
ax_x.set_title("Shift the old plan and append a terminal feedback move")
ax_x.legend(loc="best")

ax_u.axhline(u_limit, color="0.25", linestyle="--", label="input limits")
ax_u.axhline(-u_limit, color="0.25", linestyle="--")
ax_u.step(np.arange(N), u_pred, where="post", label="original inputs")
ax_u.step(np.arange(1, N + 1), u_shifted, where="post", label="shifted candidate inputs")
ax_u.set_xlabel("time index")
ax_u.set_ylabel("input $u$")
ax_u.set_ylim(-1.15, 1.15)
ax_u.legend(loc="best")

fig.tight_layout()
plt.show()

The important idea is constructive: the next MPC problem is feasible because we can exhibit a feasible candidate input sequence. This simulation illustrates the mechanism. The proof requires invariance of the terminal set under the terminal controller.

## 4. What can go wrong without terminal ingredients?

Recursive feasibility is not automatic. It is designed.

To see the issue, use a scalar unstable system

\begin{align}
x_{k+1} = 1.3x_k + u_k,
\end{align}

with the same style of box constraints. A one-step tracking controller without terminal ingredients can drive the state toward the edge of what is controllable. Once the state is too close to the upper bound, even the largest negative input cannot keep the next state feasible.

We compare this with a finite-horizon controller that enforces $x_N \in X_f$. The optimization below is intentionally transparent and small.

In [ ]:
a = 1.3
x_limit_bad = 5.0
u_limit_bad = 1.0
target = 4.0
terminal_bad = (-1.0, 1.0)

def predict_scalar(x0, U):
    xs = [float(x0)]
    for u in U:
        xs.append(a * xs[-1] + u)
    return np.array(xs)

def solve_small_mpc(x0, horizon, terminal_constraint=False):
    def cost(U):
        xs = predict_scalar(x0, U)
        return np.sum((xs[1:] - target) ** 2) + 0.02 * np.sum(np.asarray(U) ** 2)

    constraints = []
    for j in range(1, horizon + 1):
        constraints.append({"type": "ineq", "fun": lambda U, j=j: x_limit_bad - predict_scalar(x0, U)[j]})
        constraints.append({"type": "ineq", "fun": lambda U, j=j: predict_scalar(x0, U)[j] + x_limit_bad})

    if terminal_constraint:
        constraints.append({"type": "ineq", "fun": lambda U: terminal_bad[1] - predict_scalar(x0, U)[-1]})
        constraints.append({"type": "ineq", "fun": lambda U: predict_scalar(x0, U)[-1] - terminal_bad[0]})

    guess = -np.ones(horizon) if terminal_constraint else np.zeros(horizon)
    result = minimize(
        cost,
        guess,
        bounds=[(-u_limit_bad, u_limit_bad)] * horizon,
        constraints=constraints,
        method="SLSQP",
        options={"ftol": 1e-10, "maxiter": 200, "disp": False},
    )

    if not result.success:
        return None, None, False

    xs = predict_scalar(x0, result.x)
    tolerance = 2e-6
    state_ok = np.all(np.abs(xs[1:]) <= x_limit_bad + tolerance)
    terminal_ok = True
    if terminal_constraint:
        terminal_ok = terminal_bad[0] - tolerance <= xs[-1] <= terminal_bad[1] + tolerance
    return float(result.x[0]), xs, bool(state_ok and terminal_ok)

def closed_loop_rollout(x0, horizon, terminal_constraint, steps=10):
    xs = [float(x0)]
    us = []
    feasible = []
    planned = []

    for _ in range(steps):
        u, plan, ok = solve_small_mpc(xs[-1], horizon, terminal_constraint)
        feasible.append(ok)
        planned.append(plan)
        if not ok:
            break
        us.append(u)
        xs.append(a * xs[-1] + u)

    return np.array(xs), np.array(us), feasible, planned

x0_bad = 2.0
xs_no_terminal, us_no_terminal, feasible_no_terminal, _ = closed_loop_rollout(
    x0_bad, horizon=1, terminal_constraint=False, steps=10
)
xs_with_terminal, us_with_terminal, feasible_with_terminal, _ = closed_loop_rollout(
    x0_bad, horizon=4, terminal_constraint=True, steps=10
)

first_failure = None
for k, ok in enumerate(feasible_no_terminal):
    if not ok:
        first_failure = k
        break

fig, (ax_x, ax_u) = plt.subplots(2, 1, figsize=(8, 6.4), sharex=False)

ax_x.axhline(x_limit_bad, color="0.25", linestyle="--", label="state limits")
ax_x.axhline(-x_limit_bad, color="0.25", linestyle="--")
ax_x.axhspan(terminal_bad[0], terminal_bad[1], color="tab:green", alpha=0.14, label="$X_f$")
ax_x.plot(np.arange(len(xs_no_terminal)), xs_no_terminal, "o-", label="no terminal ingredients, N=1")
ax_x.plot(np.arange(len(xs_with_terminal)), xs_with_terminal, "s-", label="terminal constraint, N=4")
if first_failure is not None:
    ax_x.axvline(first_failure, color="tab:red", linestyle=":", label="first infeasible solve")
ax_x.set_xlabel("closed-loop step $k$")
ax_x.set_ylabel("state $x_k$")
ax_x.set_title("A short-sighted feasible plan can lead to a future infeasible state")
ax_x.legend(loc="best")

ax_u.axhline(u_limit_bad, color="0.25", linestyle="--", label="input limits")
ax_u.axhline(-u_limit_bad, color="0.25", linestyle="--")
ax_u.step(np.arange(len(us_no_terminal)), us_no_terminal, where="post", label="no terminal ingredients")
ax_u.step(np.arange(len(us_with_terminal)), us_with_terminal, where="post", label="terminal constraint")
ax_u.set_xlabel("closed-loop step $k$")
ax_u.set_ylabel("input $u_k$")
ax_u.set_ylim(-1.15, 1.15)
ax_u.legend(loc="best")

fig.tight_layout()
plt.show()

print("No-terminal feasibility flags:", feasible_no_terminal)
print("Terminal-constrained feasibility flags:", feasible_with_terminal)

The terminal-constrained controller is not magically better because of this one simulation. The point is the design mechanism: the terminal constraint asks the finite-horizon optimizer to end in a region where a local controller can safely continue. This simulation illustrates the mechanism. A recursive-feasibility proof still requires an invariant terminal set and a shifted-sequence argument.

## 5. Terminal cost level sets in 2D

Now consider the discrete-time double integrator

\begin{align}
A = \begin{bmatrix}1 & 1\\0 & 1\end{bmatrix},
\qquad
B = \begin{bmatrix}0\\1\end{bmatrix},
\end{align}

with

\begin{align}
Q = I, \qquad R = 1.
\end{align}

The infinite-horizon LQR terminal cost is

\begin{align}
V_f(x) = x^T P_\infty x,
\end{align}

where $P_\infty$ solves the discrete algebraic Riccati equation.

In [ ]:
A = np.array([[1.0, 1.0], [0.0, 1.0]])
B = np.array([[0.0], [1.0]])
Q = np.eye(2)
R = np.array([[1.0]])

P_inf = solve_discrete_are(A, B, Q, R)
K_lqr = np.linalg.solve(R + B.T @ P_inf @ B, B.T @ P_inf @ A)

p_grid = np.linspace(-4.5, 4.5, 260)
v_grid = np.linspace(-3.5, 3.5, 220)
P_mesh, V_mesh = np.meshgrid(p_grid, v_grid)
X_flat = np.vstack([P_mesh.ravel(), V_mesh.ravel()])
V_level = np.sum(X_flat * (P_inf @ X_flat), axis=0).reshape(P_mesh.shape)
U_lqr = -(K_lqr @ X_flat).reshape(P_mesh.shape)

x = np.array([3.2, 1.5])
trajectory = [x.copy()]
for _ in range(12):
    u = float(-(K_lqr @ x.reshape(-1, 1))[0, 0])
    x = A @ x + B[:, 0] * u
    trajectory.append(x.copy())
trajectory = np.array(trajectory)

fig, ax = plt.subplots(figsize=(7.2, 6.0))
levels = [0.5, 1.0, 2.0, 4.0, 8.0, 16.0, 32.0]
contours = ax.contour(P_mesh, V_mesh, V_level, levels=levels, colors="tab:blue")
ax.clabel(contours, inline=True, fontsize=8, fmt="%.1f")

ax.contourf(P_mesh, V_mesh, np.abs(U_lqr), levels=[0, 1.0], colors=["tab:orange"], alpha=0.12)
ax.contour(P_mesh, V_mesh, np.abs(U_lqr), levels=[1.0], colors="tab:orange", linestyles="--")

box_p = [-4, 4, 4, -4, -4]
box_v = [-3, -3, 3, 3, -3]
ax.plot(box_p, box_v, color="0.15", linestyle="--", label="state constraint box")
ax.plot(trajectory[:, 0], trajectory[:, 1], "o-", color="tab:red", label="unconstrained LQR trajectory")
ax.plot([], [], color="tab:orange", linestyle="--", label="$|u=-Kx|=1$")

ax.set_xlabel("position")
ax.set_ylabel("velocity")
ax.set_title(r"Terminal cost level sets $x^T P_\infty x$")
ax.set_aspect("equal", adjustable="box")
ax.legend(loc="upper right")
plt.show()

print("P_inf =")
print(P_inf)
print("K_lqr =", K_lqr)

The terminal cost is not only a number. It defines a geometry in state space. Its level sets show which combinations of position and velocity have equal local value under the LQR approximation.

## 6. Terminal set as a safe local-control region

A common terminal set shape is an ellipsoid

\begin{align}
X_f(\alpha) = \{x \mid x^T P_\infty x \le \alpha\}.
\end{align}

Inside this set, we use the local terminal controller

\begin{align}
u = -Kx.
\end{align}

For this teaching example, we choose $\alpha$ by sampling. The sampled points inside the ellipsoid must satisfy the state constraints, satisfy the input constraint, and stay inside the ellipsoid after one LQR step.

Sampling helps us see the idea. A mathematical proof would use exact set-invariance arguments or certified bounds, not only a grid.

In [ ]:
position_limit = 4.0
velocity_limit = 3.0
u_limit_2d = 1.0

p_samples = np.linspace(-4.5, 4.5, 181)
v_samples = np.linspace(-3.5, 3.5, 161)
P_samp, V_samp = np.meshgrid(p_samples, v_samples)
X_samp = np.vstack([P_samp.ravel(), V_samp.ravel()])

values = np.sum(X_samp * (P_inf @ X_samp), axis=0)
u_samp = -(K_lqr @ X_samp).ravel()
X_next = A @ X_samp + B @ u_samp.reshape(1, -1)
next_values = np.sum(X_next * (P_inf @ X_next), axis=0)

candidate_alphas = np.linspace(0.1, 30.0, 600)
alpha = candidate_alphas[0]
for candidate in candidate_alphas:
    inside = values <= candidate
    if not np.any(inside):
        continue
    state_ok = np.all(np.abs(X_samp[0, inside]) <= position_limit + 1e-12) and np.all(
        np.abs(X_samp[1, inside]) <= velocity_limit + 1e-12
    )
    input_ok = np.all(np.abs(u_samp[inside]) <= u_limit_2d + 1e-12)
    invariant_ok = np.all(next_values[inside] <= candidate + 1e-10)
    if state_ok and input_ok and invariant_ok:
        alpha = candidate
    else:
        break

inside_alpha = values <= alpha
near_alpha = (values > alpha) & (values <= 1.35 * alpha)
near_bad = near_alpha & (
    (np.abs(X_samp[0]) > position_limit)
    | (np.abs(X_samp[1]) > velocity_limit)
    | (np.abs(u_samp) > u_limit_2d)
    | (next_values > values + 1e-10)
)

direction = np.array([1.0, 0.7])
scale = np.sqrt(0.75 * alpha / (direction @ P_inf @ direction))
x = scale * direction
terminal_trajectory = [x.copy()]
terminal_inputs = []
for _ in range(14):
    u = float(-(K_lqr @ x.reshape(-1, 1))[0, 0])
    terminal_inputs.append(u)
    x = A @ x + B[:, 0] * u
    terminal_trajectory.append(x.copy())
terminal_trajectory = np.array(terminal_trajectory)

fig, ax = plt.subplots(figsize=(7.2, 6.0))
ax.scatter(X_samp[0, inside_alpha], X_samp[1, inside_alpha], s=5, color="tab:green", alpha=0.18, label=r"sampled points inside $X_f(\alpha)$")
ax.scatter(X_samp[0, near_bad], X_samp[1, near_bad], s=5, color="tab:red", alpha=0.18, label="nearby sampled violations")
ax.contour(P_mesh, V_mesh, V_level, levels=[alpha], colors="tab:green", linewidths=2.5)
ax.plot(box_p, box_v, color="0.15", linestyle="--", label="state constraint box")
ax.contour(P_mesh, V_mesh, np.abs(U_lqr), levels=[u_limit_2d], colors="tab:orange", linestyles="--")
ax.plot(terminal_trajectory[:, 0], terminal_trajectory[:, 1], "o-", color="tab:blue", label="local LQR trajectory")

ax.set_xlabel("position")
ax.set_ylabel("velocity")
ax.set_title(f"Sampled terminal ellipsoid, alpha = {alpha:.2f}")
ax.set_aspect("equal", adjustable="box")
ax.legend(loc="upper right", fontsize=8)
plt.show()

print(f"Selected alpha by sampling: {alpha:.3f}")
print(f"Largest sampled |u=-Kx| inside ellipsoid: {np.max(np.abs(u_samp[inside_alpha])):.3f}")
print(f"Largest sampled next V / alpha inside ellipsoid: {np.max(next_values[inside_alpha] / alpha):.3f}")

Inside the terminal set, a simple local controller is known to keep the system safe. MPC only needs to drive the state into this region.

The plot should be read as geometric intuition. The actual guarantee comes from proving that the terminal set is constraint-admissible and invariant under the terminal feedback law.

## 7. Stability intuition via Lyapunov decrease

For the unconstrained LQR feedback $u=-Kx$, the value

\begin{align}
V(x_k) = x_k^T P_\infty x_k
\end{align}

decreases along closed-loop trajectories. This is the Lyapunov picture behind the terminal cost.

In [ ]:
x = np.array([3.0, -1.0])
lqr_states = [x.copy()]
lqr_inputs = []
lqr_values = [float(x @ P_inf @ x)]

for _ in range(18):
    u = float(-(K_lqr @ x.reshape(-1, 1))[0, 0])
    lqr_inputs.append(u)
    x = A @ x + B[:, 0] * u
    lqr_states.append(x.copy())
    lqr_values.append(float(x @ P_inf @ x))

lqr_states = np.array(lqr_states)
lqr_inputs = np.array(lqr_inputs)
lqr_values = np.array(lqr_values)
delta_values = np.diff(lqr_values)

fig, axes = plt.subplots(4, 1, figsize=(8, 8.8), sharex=False)

axes[0].plot(lqr_states[:, 0], "o-", label="position")
axes[0].plot(lqr_states[:, 1], "s-", label="velocity")
axes[0].set_ylabel("state")
axes[0].set_title("Closed-loop LQR trajectory")
axes[0].legend(loc="best")

axes[1].step(np.arange(len(lqr_inputs)), lqr_inputs, where="post", color="tab:orange")
axes[1].set_ylabel("input $u_k$")

axes[2].plot(lqr_values, "o-", color="tab:green")
axes[2].set_ylabel("$V(x_k)$")

axes[3].axhline(0.0, color="0.25", linestyle="--")
axes[3].bar(np.arange(len(delta_values)), delta_values, color="tab:red", alpha=0.75)
axes[3].set_ylabel(r"$\Delta V_k$")
axes[3].set_xlabel("time step $k$")

fig.tight_layout()
plt.show()

print("All plotted Delta V values are non-positive:", np.all(delta_values <= 1e-10))

Stability is connected to a decreasing value function. Terminal cost and terminal set are chosen so that the finite-horizon MPC problem inherits this behavior.

Again, the simulation illustrates the mechanism. The stability proof is a Lyapunov argument: one shows that the optimal finite-horizon value decreases by at least a positive stage cost under the shifted candidate policy.

## 8. Optional interactive mini-demo

An interactive version could later be added here.

A useful small extension would add sliders for horizon length, terminal-set size, and input limit, then redraw the scalar predecessor interval on a line. This notebook keeps the static figures so it remains dependency-free and easy to execute in a plain Python environment.